# 主题：Pandas怎样新增数据列
# 一、直接赋
- 如果想要修改某一列，我们可以按以下形式进行"df.loc[:, "column_name"] = ..."
- 如果想要新增某一列，方法同上
- Pandas支持广播操作。广播（broadcasting）是一种在NumPy和Pandas等数组和矩阵处理库中常见的概念。它指的是在对不同形状的数组进行算术运算时，系统会自动进行一系列的操作，使得它们能够兼容并执行相应的运算，而无需用户显式地进行形状匹配或循环操作。

- 广播操作通常包括以下步骤：

1. 维度补齐（Dimensionality Expansion）：当两个数组的维度不同时，系统会自动地在较小的数组形状的前面加上长度为1的维度，使得它们的维度数相等。

2. 形状匹配（Shape Matching）：系统会比较两个数组的形状，在维度长度为1的维度上进行形状匹配，使得两个数组在这些维度上具有相同的长度。

3. 值复制（Value Replication）：如果某个维度的长度不为1，但两个数组在该维度上的长度不同，系统会复制数组中的值，使得它们在该维度上的长度相等。

4. 逐元素操作（Element-wise Operation）：最终，系统对广播后的数组执行逐元素的算术运算，得到最终的结果。

5. 广播操作的优势在于它使得处理不同形状的数组变得更加简洁和高效，同时提高了代码的可读性和可维护性。

In [2]:
import pandas as pd
file_path = r"D:\JupyterProject\study-pandas\file\datas\weather_20230115134249.csv"
df = pd.read_csv(file_path)
print(df.head())

           日期   城市  行政区  观测站  气温(度)  相对湿度(%)  累积雨量(mm)
0  2015-01-01  新北市  烏來區   福山  13.7℃       92       0.0
1  2015-01-02  臺南市  安平區   安平  23.5℃       70       0.0
2  2015-01-03  臺東縣  東河鄉  七塊厝  19.6℃       86       0.0
3  2015-01-04  新北市  貢寮區   福隆  14.2℃       96     -99.0
4  2015-01-05  南投縣  仁愛鄉  小奇萊   8.3℃       57       0.0


比如我现在想把气温的后缀C去掉，这样才方便我进行一些数值上的处理。

In [6]:
#替换温度后面的
df.loc[:, "气温(度)"] = df["气温(度)"].str.replace("℃", "").astype('float32')

In [7]:
print(df.head())

           日期   城市  行政区  观测站  气温(度)  相对湿度(%)  累积雨量(mm)
0  2015-01-01  新北市  烏來區   福山   13.7       92       0.0
1  2015-01-02  臺南市  安平區   安平   23.5       70       0.0
2  2015-01-03  臺東縣  東河鄉  七塊厝   19.6       86       0.0
3  2015-01-04  新北市  貢寮區   福隆   14.2       96     -99.0
4  2015-01-05  南投縣  仁愛鄉  小奇萊    8.3       57       0.0


In [7]:
#新增一列
df.loc[:, "华氏度"] = df["气温(度)"] + 273.15 #自动进行广播操作
df.head()

,日期,城市,行政区,观测站,气温(度),相对湿度(%),累积雨量(mm),low_temp,华氏度
0,2015-01-01,新北市,烏來區,福山,13.7,92,0.0,-12.837816,286.850006
1,2015-01-02,臺南市,安平區,安平,23.5,70,0.0,-17.254067,296.649994
2,2015-01-03,臺東縣,東河鄉,七塊厝,19.6,86,0.0,-12.044249,292.750000
3,2015-01-04,新北市,貢寮區,福隆,14.2,96,-99.0,-12.853192,287.350006
4,2015-01-05,南投縣,仁愛鄉,小奇萊,8.3,57,0.0,-11.776900,281.449982


# 二、df.apply方法
## 2.1 用法
- df.apply(func, axis=0)：沿着指定的列（轴 0）应用函数 func，其中 func 是一个可以接受一个Series对象作为输入并返回一个标量值的函数。
- df.apply(func, axis=1)：沿着指定的行（轴 1）应用函数 func，其中 func 是一个可以接受一个Series对象作为输入并返回一个标量值的函数。
- func 可以是一个自定义函数、lambda函数或者是已经存在的函数。

## 2.2 关于axis参数的理解

- 理解 axis 参数是使用 df.apply() 方法时的关键。它指定了函数在DataFrame的哪个轴上被应用。这里来详细解释一下 axis 参数的区别：

- 当 axis=0 时，函数会沿着DataFrame的列方向进行应用。这意味着函数将会在每一列上被调用，即每次调用时，传入函数的是这一列的所有元素。
通常在这种情况下，你会对每一列进行某种操作，比如计算每列的总和、平均值等。
如果你有一个函数，该函数接受一个Series对象作为参数，并返回一个标量值，那么你可以使用 axis=0。

- 当 axis=1 时，函数会沿着DataFrame的行方向进行应用。这意味着函数将会在每一行上被调用，即每次调用时，传入函数的是这一行的所有元素。
通常在这种情况下，你会对每一行进行某种操作，比如根据行的值计算新的列，或者过滤行。
如果你有一个函数，该函数接受一个Series对象作为参数，并返回一个标量值，那么你可以使用 axis=1。

实例如下：添加一个温度类型列

In [8]:
def get_temp_type(row):
    if row["气温(度)"] > 30:
        return "高温"
    elif row["气温(度)"] >= 0:
        return "常温"
    return "低温"

df.loc[:, "温度类型"] = df.apply(get_temp_type, axis = 1)
print(df.head())
print(df.loc[:, "温度类型"].value_counts())
print(df["温度类型"].value_counts())

           日期   城市  行政区  观测站  气温(度)  相对湿度(%)  累积雨量(mm)   low_temp         华氏度  \
0  2015-01-01  新北市  烏來區   福山   13.7       92       0.0 -12.837816  286.850006   
1  2015-01-02  臺南市  安平區   安平   23.5       70       0.0 -17.254067  296.649994   
2  2015-01-03  臺東縣  東河鄉  七塊厝   19.6       86       0.0 -12.044249  292.750000   
3  2015-01-04  新北市  貢寮區   福隆   14.2       96     -99.0 -12.853192  287.350006   
4  2015-01-05  南投縣  仁愛鄉  小奇萊    8.3       57       0.0 -11.776900  281.449982   

  温度类型  
0   常温  
1   常温  
2   常温  
3   常温  
4   常温  
常温    472
低温     11
高温      1
Name: 温度类型, dtype: int64
常温    472
低温     11
高温      1
Name: 温度类型, dtype: int64


# 三、df.assign方法
## 3.1 用法
- assign方法与apply方法不同，assign会返回一个新的对象，并且可以同时增加多个列。
- 如果列使用了匿名函数，那么传给匿名函数作为参数的是原DataFrame表。
## 3.2 实例

In [18]:
new_df = df.assign(湿度类型 = lambda row: row["相对湿度(%)"] > 60)
print(new_df.head())

           日期   城市  行政区  观测站  气温(度)  相对湿度(%)  累积雨量(mm)         华氏度 温度类型   湿度类型
0  2015-01-01  新北市  烏來區   福山   13.7       92       0.0  286.850006   常温   True
1  2015-01-02  臺南市  安平區   安平   23.5       70       0.0  296.649994   常温   True
2  2015-01-03  臺東縣  東河鄉  七塊厝   19.6       86       0.0  292.750000   常温   True
3  2015-01-04  新北市  貢寮區   福隆   14.2       96     -99.0  287.350006   常温   True
4  2015-01-05  南投縣  仁愛鄉  小奇萊    8.3       57       0.0  281.449982   常温  False


In [45]:
column_name = "湿度类型"
column_value = lambda row: row["相对湿度(%)"] > 60
new_df = df.assign(**{column_name: column_value}) #这里的解包操作相当于将其变为关键字参数的形式。

# 四、按条件选择分组分别赋值

In [4]:
#生成一个新列 最低温度
import numpy as np
df['low_temp'] = np.random.uniform(-20, -10, size = len(df))
print(df.head())

           日期   城市  行政区  观测站  气温(度)  相对湿度(%)  累积雨量(mm)   low_temp
0  2015-01-01  新北市  烏來區   福山  13.7℃       92       0.0 -12.837816
1  2015-01-02  臺南市  安平區   安平  23.5℃       70       0.0 -17.254067
2  2015-01-03  臺東縣  東河鄉  七塊厝  19.6℃       86       0.0 -12.044249
3  2015-01-04  新北市  貢寮區   福隆  14.2℃       96     -99.0 -12.853192
4  2015-01-05  南投縣  仁愛鄉  小奇萊   8.3℃       57       0.0 -11.776900


In [24]:
#新建一个温差类型
df["wencha_type"] = ""

df.loc[df['气温(度)'] - df['low_temp'] > 10, 'wencha_type'] = "温差大"
df.loc[df['气温(度)'] - df['low_temp'] <= 10, 'wencha_type'] = "温差小"
print(df.head())

           日期   城市  行政区  观测站  气温(度)  相对湿度(%)  累积雨量(mm)         华氏度 温度类型  \
0  2015-01-01  新北市  烏來區   福山   13.7       92       0.0  286.850006   常温   
1  2015-01-02  臺南市  安平區   安平   23.5       70       0.0  296.649994   常温   
2  2015-01-03  臺東縣  東河鄉  七塊厝   19.6       86       0.0  292.750000   常温   
3  2015-01-04  新北市  貢寮區   福隆   14.2       96     -99.0  287.350006   常温   
4  2015-01-05  南投縣  仁愛鄉  小奇萊    8.3       57       0.0  281.449982   常温   

    low_temp wencha_type  
0 -16.214767         温差大  
1 -18.687733         温差大  
2 -10.844213         温差大  
3 -18.517008         温差大  
4 -19.665756         温差大  


In [2]:
#如果使用assign方法呢？
name = "温差类型"

new_df = df.assign(**{name :["温差大" if cell > 30 else "温差小" for cell in df['气温(度)'] - df['low_temp']]})
print(new_df.head())

NameError: name 'df' is not defined